# Lab 5 FDSAI (2025-2026) – Gamma distribution: verification, method of moments, maximum likelihood
Run all cells from top to bottom. Notation: $X \sim \Gamma(k, b)$ with shape $k>0$ and scale $b>0$, and density

$$f(x)=\frac{1}{\Gamma(k)\,b^{k}}\,x^{k-1}e^{-x/b},\qquad x>0 .$$

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy import stats, integrate, optimize, special

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

## Exercise 1a – Verify that f is a probability density function
A function $f$ is a probability density function if:
1. $f(x)\ge 0$ for all $x$;
2. $\int_{-\infty}^{\infty} f(x)\,dx = 1$.

**Condition 1.** For $x>0$: $x^{k-1}>0$, $e^{-x/b}>0$, $\Gamma(k)>0$ and $b^k>0$, so $f(x)>0$. (For $x\le 0$ the density is 0.)

**Condition 2.** Substitute $t = x/b$, so $x = bt$ and $dx = b\,dt$:

$$\int_0^\infty \frac{x^{k-1}e^{-x/b}}{\Gamma(k)b^k}\,dx=\frac{1}{\Gamma(k)b^k}\int_0^\infty (bt)^{k-1}e^{-t}\,b\,dt=\frac{b^{k}}{\Gamma(k)b^k}\int_0^\infty t^{k-1}e^{-t}\,dt=\frac{\Gamma(k)}{\Gamma(k)}=1,$$

because $\int_0^\infty t^{k-1}e^{-t}dt=\Gamma(k)$ is the definition of the Gamma function. The code below checks the same result symbolically and numerically.

In [ ]:
# Symbolic check with SymPy
x, k, b = sp.symbols("x k b", positive=True)
f = x ** (k - 1) * sp.exp(-x / b) / (sp.gamma(k) * b ** k)       # Gamma(k, b) density

integral_unnormalised = sp.integrate(x ** (k - 1) * sp.exp(-x / b), (x, 0, sp.oo))
print("Integral of x^(k-1) e^(-x/b) over (0, oo) =", integral_unnormalised)      # b^k * Gamma(k)

total = sp.simplify(integral_unnormalised / (sp.gamma(k) * b ** k))
print("Integral of f over (0, oo) =", total)                                      # must be 1

In [ ]:
# Numerical check for several parameter pairs (k, b)
def gamma_pdf(x, k, b):
    # Density written exactly as in the lab sheet (gammaln avoids overflow for large k)
    return np.exp((k - 1) * np.log(x) - x / b - special.gammaln(k) - k * np.log(b))

rows = []
for k_val, b_val in [(0.5, 1.0), (1.0, 2.0), (2.0, 1.0), (3.0, 2.0), (5.0, 0.5), (9.0, 1.0)]:
    area, err = integrate.quad(gamma_pdf, 0, np.inf, args=(k_val, b_val))
    grid = np.linspace(1e-6, 60, 20000)
    min_f = gamma_pdf(grid, k_val, b_val).min()
    max_diff = np.abs(gamma_pdf(grid, k_val, b_val) - stats.gamma.pdf(grid, a=k_val, scale=b_val)).max()
    rows.append((k_val, b_val, area, min_f, max_diff))

print(f"{'k':>5} {'b':>5} {'integral':>12} {'min f(x)':>12} {'max |f - scipy pdf|':>22}")
for r in rows:
    print(f"{r[0]:5.1f} {r[1]:5.1f} {r[2]:12.8f} {r[3]:12.3e} {r[4]:22.3e}")

In [ ]:
# Graph of the density for several parameters
xs = np.linspace(0.01, 15, 500)
fig, ax = plt.subplots(figsize=(8, 4.5))
for k_val, b_val in [(1, 2), (2, 2), (3, 2), (5, 1), (9, 0.5)]:
    ax.plot(xs, gamma_pdf(xs, k_val, b_val), linewidth=2, label=f"k = {k_val}, b = {b_val}")
ax.set_title("Density of the Gamma(k, b) distribution")
ax.set_xlabel("x"); ax.set_ylabel("f(x)"); ax.set_ylim(0, 1.0)
ax.grid(True, linestyle="--", alpha=0.35); ax.legend()
plt.tight_layout(); plt.show()

**Conclusion 1a.** $f\ge 0$ everywhere and its integral over $(0,\infty)$ equals 1 for every $k>0$, $b>0$ (checked analytically, symbolically and numerically), so $f$ is a probability density function.

## Exercise 1b – Method of moments for k (with b = 1)
The method of moments equates the theoretical moment with the sample moment. For $X\sim\Gamma(k,b)$:

$$E[X]=\int_0^\infty x\,\frac{x^{k-1}e^{-x/b}}{\Gamma(k)b^k}dx=\frac{b^{k+1}\Gamma(k+1)}{\Gamma(k)b^k}=k\,b .$$

With $b=1$ we have $E[X]=k$. Setting $E[X]=\bar{x}$ gives the estimator

$$\boxed{\hat{k}_{MM}=\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i}$$

Sample: $x_1=3.56,\ x_2=1.64,\ x_3=2.77,\ x_4=2.62,\ x_5=3.08,\ x_6=1.91$.

In [ ]:
sample = np.array([3.56, 1.64, 2.77, 2.62, 3.08, 1.91])
n = len(sample)
b_known = 1.0

x_bar = sample.mean()
k_mm = x_bar / b_known           # from E[X] = k * b = x_bar
print(f"n = {n}, sum = {sample.sum():.2f}")
print(f"Sample mean = {x_bar:.4f}")
print(f"Method of moments estimate: k_hat = x_bar / b = {k_mm:.4f}")

In [ ]:
# Check: the theoretical mean of Gamma(k_hat, 1) equals the sample mean
fitted = stats.gamma(a=k_mm, scale=b_known)
print(f"Theoretical mean of Gamma(k_hat, 1) = {fitted.mean():.4f}  (sample mean = {x_bar:.4f})")
print(f"Theoretical variance = {fitted.var():.4f}  (sample variance = {sample.var(ddof=1):.4f}; not used in the estimation)")

# Fitted density versus the sample
xs = np.linspace(0.01, 8, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(sample, bins=np.arange(1, 4.5, 0.5), density=True, color="lightsteelblue", edgecolor="white", label="Sample (n = 6)")
ax.plot(xs, fitted.pdf(xs), color="tab:red", linewidth=2, label=f"Gamma(k = {k_mm:.3f}, b = 1)")
ax.plot(sample, np.zeros_like(sample), "|", color="black", markersize=18, label="Observations")
ax.axvline(x_bar, color="gray", linestyle="--", label=f"Sample mean = {x_bar:.3f}")
ax.set_title("Method of moments fit"); ax.set_xlabel("x"); ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.35); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Extra (not required): maximum likelihood estimate of k for b = 1, for comparison.
# The likelihood equation is digamma(k) = mean(ln x_i), solved numerically.
mean_log = np.log(sample).mean()
k_mle = optimize.brentq(lambda kk: special.digamma(kk) - mean_log, 0.01, 100)
print(f"MLE of k (b = 1): {k_mle:.4f}   vs   method of moments: {k_mm:.4f}")

**Conclusion 1b.** Since $E[X]=kb$ and $b=1$, the method of moments estimator is $\hat{k}=\bar{x}$. For the given sample $\bar{x}=15.58/6\approx 2.5967$, so **$\hat{k}_{MM}\approx 2.60$**. (The maximum likelihood estimate computed as an extra check is different, which is normal: the two methods are different estimators and, with only 6 observations, they do not have to coincide.)

## Exercise 2a – Maximum likelihood estimator of b
Let $x_1,\dots,x_n$ be an i.i.d. sample from $\Gamma(k,b)$ (the shape $k$ is treated as known, the parameter to estimate is $b$).

**Likelihood:**
$$L(b)=\prod_{i=1}^{n}\frac{x_i^{k-1}e^{-x_i/b}}{\Gamma(k)b^k}=\frac{\left(\prod x_i\right)^{k-1}e^{-\sum x_i/b}}{\Gamma(k)^n\,b^{nk}}$$

**Log-likelihood:**
$$\ell(b)=-n\ln\Gamma(k)-nk\ln b+(k-1)\sum_{i=1}^{n}\ln x_i-\frac{1}{b}\sum_{i=1}^{n}x_i$$

**First-order condition:**
$$\frac{d\ell}{db}=-\frac{nk}{b}+\frac{\sum x_i}{b^2}=0\;\Longrightarrow\; \boxed{\hat{b}_{ML}=\frac{\sum_{i=1}^n x_i}{nk}=\frac{\bar{x}}{k}}$$

**Second-order condition:** $\dfrac{d^2\ell}{db^2}=\dfrac{nk}{b^2}-\dfrac{2\sum x_i}{b^3}$; at $\hat b=\bar x/k$ it equals $-\dfrac{n k^3}{\bar{x}^2}<0$, so $\hat{b}$ is a maximum. The code below repeats the derivation symbolically.

In [ ]:
# Symbolic derivation of the maximum likelihood estimator of b
n_s, k_s, b_s = sp.symbols("n k b", positive=True)
S, Lsum = sp.symbols("S L", positive=True)      # S = sum of x_i, L = sum of ln(x_i)

loglik = -n_s * sp.log(sp.gamma(k_s)) - n_s * k_s * sp.log(b_s) + (k_s - 1) * Lsum - S / b_s
print("Log-likelihood:      l(b) =", loglik)

score = sp.diff(loglik, b_s)                    # first derivative
print("First derivative:    dl/db =", sp.simplify(score))

b_hat = sp.solve(sp.Eq(score, 0), b_s)[0]       # solve dl/db = 0
print("Solution:            b_hat =", b_hat, "   (= x_bar / k, since S = n * x_bar)")

second = sp.diff(loglik, b_s, 2).subs(b_s, b_hat)
print("Second derivative at b_hat =", sp.simplify(second), "  (negative -> maximum)")

### Numerical check with simulated data
To see the estimator at work, a sample is generated from a Gamma distribution with known parameters ($k=3$, $b=2$). The closed-form estimator $\bar{x}/k$ is compared with the maximum of the log-likelihood found by a numerical optimiser.

In [ ]:
rng = np.random.default_rng(42)       # fixed seed for reproducible results
k_true, b_true, n_sim = 3.0, 2.0, 500
data = rng.gamma(shape=k_true, scale=b_true, size=n_sim)

# Closed-form maximum likelihood estimator derived above
b_closed = data.mean() / k_true

# Numerical maximisation of the log-likelihood (minimise the negative log-likelihood)
def neg_loglik(b_val):
    return -np.sum(stats.gamma.logpdf(data, a=k_true, scale=b_val))

opt = optimize.minimize_scalar(neg_loglik, bounds=(0.01, 20), method="bounded")

print(f"True b                    : {b_true}")
print(f"Closed-form MLE  x_bar / k: {b_closed:.4f}")
print(f"Numerical maximum of l(b) : {opt.x:.4f}")

In [ ]:
# Log-likelihood as a function of b: the maximum is at x_bar / k
bs = np.linspace(0.8, 4, 400)
ll = np.array([-neg_loglik(v) for v in bs])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(bs, ll, linewidth=2, color="tab:blue", label="Log-likelihood l(b)")
ax.axvline(b_closed, color="tab:red", linestyle="--", label=f"b_hat = x_bar/k = {b_closed:.3f}")
ax.axvline(b_true, color="black", linestyle=":", label=f"True b = {b_true}")
ax.set_title("Log-likelihood of b (k known)"); ax.set_xlabel("b"); ax.set_ylabel("l(b)")
ax.grid(True, linestyle="--", alpha=0.35); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# The estimator is consistent: the error decreases as the sample size grows
rows = []
for size in [10, 50, 200, 1000, 10000]:
    estimates = [rng.gamma(k_true, b_true, size).mean() / k_true for _ in range(500)]   # 500 repetitions
    rows.append((size, np.mean(estimates), np.std(estimates)))

print(f"{'n':>7} {'mean of b_hat':>15} {'sd of b_hat':>13}")
for r in rows:
    print(f"{r[0]:7d} {r[1]:15.4f} {r[2]:13.4f}")

**Conclusion 2a.** The maximum likelihood estimator of the scale parameter (with $k$ known) is

$$\hat{b}_{ML}=\frac{\bar{x}}{k}=\frac{1}{nk}\sum_{i=1}^{n}x_i .$$

It is unbiased ($E[\hat b]=E[\bar X]/k=kb/k=b$) and consistent: the simulation shows that its mean stays near the true $b$ while its standard deviation decreases roughly like $1/\sqrt{n}$. Note that with $b$ estimated this way the method of moments and maximum likelihood give the same formula for $b$ when $k$ is known.